In [0]:
dbutils.widgets.text("catalog", "vp_rm20040829")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql.functions import *

In [0]:
spark.sql(f"""
create table if not exists {catalog}.ops.entitlements(
    user_email string,
    allowed_city string,
    can_view_pii boolean
)          
""")

In [0]:
spark.sql(f"GRANT USE CATALOG ON CATALOG {catalog} TO `account users`")
spark.sql(f"GRANT USE SCHEMA, SELECT ON SCHEMA {catalog}.gold TO `account users`")

In [0]:
#mask phone in gold.dim_member: real value if can_view_pii = true, otherwise XXXXXX + last 4 digits

# 1. Create a SQL UDF that checks the current user's can_view_pii entitlement
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.ops.mask_phone(phone STRING)
RETURNS STRING
RETURN CASE
  WHEN phone IS NULL THEN NULL
  WHEN EXISTS (
    SELECT 1 FROM {catalog}.ops.entitlements
    WHERE user_email = current_user() AND can_view_pii = true
  )
  THEN phone
  ELSE CONCAT('XXXXXX', RIGHT(phone, 4))
END
""")

spark.sql(f"""
ALTER TABLE {catalog}.gold.dim_members
ALTER COLUMN phone SET MASK {catalog}.ops.mask_phone
""")

In [0]:
%sql
select * from vp_rm20040829.gold.dim_members

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.ops.filter_city(city STRING)
RETURNS BOOLEAN
RETURN EXISTS (
  SELECT 1 FROM {catalog}.ops.entitlements
  WHERE user_email = current_user()
    AND (allowed_city = 'ALL' OR allowed_city = city)
)
""")

spark.sql(f"""
ALTER TABLE {catalog}.gold.agg_daily_station
SET ROW FILTER {catalog}.ops.filter_city ON (city)
""")



In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ops.checkpoint_count(
    checkpoint_name STRING,
    table_name     STRING,
    row_count      BIGINT,
    captured_at    TIMESTAMP
)
""")

# Tables to checkpoint
checkpoint_tables = [
    f"{catalog}.bronze.sessions",
    f"{catalog}.bronze.members",
    f"{catalog}.bronze.chargers",
    f"{catalog}.silver.sessions",
    f"{catalog}.silver.quarantine_sessions",
    f"{catalog}.silver.members",
    f"{catalog}.silver.chargers",
    f"{catalog}.gold.fact_sessions",
    f"{catalog}.gold.agg_daily_station",
]


def capture_checkpoint(checkpoint_name):
    """Insert one row per table with current row counts."""
    from datetime import datetime
    captured_at = datetime.now()
    rows = []
    for table_name in checkpoint_tables:
        cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {table_name}").collect()[0]["c"]
        rows.append((checkpoint_name, table_name, cnt, captured_at))
    (
        spark.createDataFrame(
            rows,
            schema="checkpoint_name string, table_name string, row_count bigint, captured_at timestamp",
        )
        .write.mode("append")
        .saveAsTable(f"{catalog}.ops.checkpoint_count")
    )
    print(f"Checkpoint '{checkpoint_name}' captured at {captured_at}")


# RUN 1 – upload batch 1 files only, run the pipeline, then:
#capture_checkpoint("AFTER_BATCH_1")

# RUN 2 – upload all batch 2 files, run the pipeline, then:
#capture_checkpoint("AFTER_BATCH_2")

# RUN 3 – no new files (rerun), run the pipeline, then:
capture_checkpoint("AFTER_RERUN")